### Code

In [ ]:
import pandas as pd

df = pd.read_csv('ocr_results.csv')

In [ ]:
df[df['image_id']=='datasets/nomnaocr/images/DVSKTT-1 Quyen thu/DVSKTT_thu_I_1b.jpg@22']


In [ ]:
df['predicted_text'].value_counts().to_csv('predicted_text_frequencies.csv')


In [ ]:
from pathlib import Path

df[['page_id', 'bbox_id']] = df['image_id'].str.split('@', expand=True)

df['page_id'] = df['page_id'].apply(lambda x: str(Path(Path(x).parent.name, Path(x).stem)))
df['bbox_id'] = df['bbox_id'].astype(int)

In [ ]:
page_id = 'DVSKTT-2 Ngoai ky toan thu/DVSKTT_ngoai_IV_9b'
df[df['page_id'] == page_id]

In [ ]:
detection_root = Path('detection/nomnaocr/labels')
detection_file = detection_root / f'{page_id}.txt'
image_root = Path('datasets/nomnaocr/images')

import imagesize
width, height = imagesize.get(image_root / f'{page_id}.jpg')
import numpy as np
# read label YOLO format
with open(detection_file, 'r') as f:
    labels = f.readlines()
classes = [label.strip().split()[0] for label in labels]
bboxes = [list(map(float, label.strip().split()[1:])) for label in labels]
classes = np.array(classes)
bboxes = np.array(bboxes)
bboxes = bboxes * np.array([width, height, width, height])
bboxes = bboxes.astype(int)


In [ ]:
def parse_line_labels(label_file):
    labels = []
    with open(label_file, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) == 9:
                try:
                    x1, y1, x2, y2, x3, y3, x4, y4 = map(float, parts[:8])
                    label = parts[8]
                    xs = [x1, x2, x3, x4]
                    ys = [y1, y2, y3, y4]
                    xmin = min(xs)
                    xmax = max(xs)
                    ymin = min(ys)
                    ymax = max(ys)
                    labels.append({
                        "bbox": (xmin, xmax, ymin, ymax),
                        "label": label
                    })
                except ValueError:
                    continue
    return labels

In [ ]:
line_labels_root = Path('datasets/nomnaocr/line_labels')

line_label_file = line_labels_root / f'{page_id}.txt'

line_labels = parse_line_labels(line_label_file)

In [ ]:
line_labels[0]


In [ ]:

# find bboxes that are inside the line label bbox
def is_inside(bbox, line_bbox):
    x, y, w, h = bbox
    x_center = x + w / 2
    y_center = y + h / 2
    lxmin, lxmax, lymin, lymax = line_bbox
    return (lxmin <= x_center <= lxmax) and (lymin <= y_center <= lymax)

bboxes_ids = [id for id in range(len(bboxes)) if is_inside(bboxes[id], line_labels[0]['bbox'])]
# sort by y center, then x center
def bbox_center(bbox):
    x, y, w, h = bbox
    return (x + w / 2, y + h / 2)
bboxes_ids = sorted(bboxes_ids, key=lambda id: (bbox_center(bboxes[id])[1], bbox_center(bboxes[id])[0]))

In [ ]:
bboxes_ids

texts = []
for bbox_id in bboxes_ids:
    row = df[(df['page_id'] == page_id) & (df['bbox_id'] == bbox_id)]
    if not row.empty:
        texts.append(row.iloc[0]['predicted_text'])
''.join(texts)

In [ ]:
line_labels[0]

### Analyze results

In [ ]:
import pandas as pd
from glob import glob
from pathlib import Path
eval_root = Path('evaluation_results_retrain_v2.1')

import editdistance

def compute_cer_row(row):
    distance = editdistance.eval(row['predicted_text'], row['ground_truth_text'])
    if (len(row['predicted_text'])  - row['variants_count']) >= len(row['ground_truth_text']):
        distance -= row['variants_count']
    return distance


def compute_cer(eval_df):
    eval_df['predicted_text'] = eval_df['predicted_text'].fillna('')
    eval_df['distance'] = eval_df.apply(lambda row: compute_cer_row(row), axis=1)
    eval_df['ground_truth_length'] = eval_df['ground_truth_text'].apply(len)
    eval_df['cer'] = eval_df['distance'] / eval_df['ground_truth_length']
    return eval_df

eval_dfs = []
for fn in eval_root.glob('**/*.csv'):
    eval_df = pd.read_csv(fn)
    eval_df = compute_cer(eval_df)
    eval_dfs.append(eval_df)
eval_df = pd.concat(eval_dfs, ignore_index=True)
# save to file
# eval_df.to_csv('combined_evaluation_results_retrain.csv', index=False)
cer = eval_df['distance'].sum() / eval_df['ground_truth_length'].sum()

eval_df['folder'] = eval_df['page_id'].apply(lambda x: Path(x).parent.name)
cer_per_folder = eval_df.groupby('folder').apply(lambda df: df['distance'].sum() / df['ground_truth_length'].sum())
cer_per_folder


### Load results

In [ ]:
import pandas as pd
eval_df = pd.read_csv('combined_evaluation_results_retrain.csv')

### CER per folder
from pathlib import Path
eval_df['folder'] = eval_df['page_id'].apply(lambda x: Path(x).parent.name)
cer_per_folder = eval_df.groupby('folder').apply(lambda df: df['distance'].sum() / df['ground_truth_length'].sum())
cer_per_folder

### check IDS vocab

In [ ]:
# read ids
from pathlib import Path
ids_list = [line.strip().split()[0] for line in open('nom-ids/ids_exp.txt').readlines()]

In [ ]:
len(ids_list)

vocab = set()
from utils import parse_line_labels

# find all txt files in line_labels folder
for path in Path('datasets/nomnaocr/line_labels').glob('**/*.txt'):
    for line_label in parse_line_labels(path):
        vocab.update(line_label['label'])
        # if '\U000f0629' in line_label['label']:
        #     print(path, line_label)
        #     break

In [ ]:
[v for v in vocab if v not in ids_list]

In [ ]:
for path in Path('datasets/nomnaocr/line_labels').glob('**/*.txt'):
    for line_label in parse_line_labels(path):
        if '\U000f0629' in line_label['label']:
            print(path, line_label)
            break

In [ ]:
with open('vocab_exp.txt', 'w', encoding='utf-8') as f:
    f.write(''.join([v for v in vocab if v not in ids_list]))

In [ ]:
ids_list_yb = [line.strip().split('\t')[0] for line in open('yb-ids/ids_lv1.txt').readlines()]  
len(ids_list_yb)

In [ ]:
len([v for v in vocab if v not in ids_list_yb])

### Header

In [ ]:
from utils import process_ocr_results, parse_line_labels, is_inside, load_yolo_bboxes
import imagesize
width, height = imagesize.get('datasets/nomnaocr/images/Tale of Kieu 1866/page024a.jpg')
_,bboxes = load_yolo_bboxes('datasets/nomnaocr/labels/Tale of Kieu 1866/page024a.txt', width, height)
parse_line_labels('datasets/nomnaocr/line_labels/Tale of Kieu 1866/page024a.txt')
# bboxes
